# Regras fuzzy que valem para qualquer rede: tipo 1 × tipo 2

**Pergunta:** as regras que detectam erro, aprendidas olhando *algumas* redes, funcionam numa rede que o sistema nunca viu?

1. Treina **três redes bem diferentes** no HAM10000, no mesmo split por lesão: EfficientNet-B0, ResNet-50 e ViT-B/16.
2. Lê cada rede por dentro (probes em cada bloco) com a **profundidade medida de 0 a 1**, para as features serem comparáveis.
3. **Deixa uma rede de fora:** aprende os sistemas fuzzy nas outras duas e aplica na que ficou de fora.
   Sistema fuzzy principal: **fuzzy aditivo** (regras de 1 condição somadas, 5 features), que nas rodadas anteriores
   ficou colado na regressão logística. Referências: só P, regressão logística e boosting.
   (O Wang-Mendel também roda, mas só para consulta: aparece no fim do notebook e em abas `interno_*`.)
   - **Tipo 1:** uma curva por termo ("taxa_trocas ALTA"), das redes de origem juntas.
   - **Tipo 2:** a curva vira uma faixa = a discordância entre as redes de origem. A saída vira um intervalo e três zonas: alarme, incerto, seguro.

**Antes de rodar:** Settings → Accelerator → **GPU**; Settings → **Internet → On** (para baixar os pesos do ImageNet).
Tempo estimado numa T4/P100: cerca de 1 a 2 horas para as três redes (o ViT é o mais lento).
Se a sessão cair, rode de novo: checkpoints e features ficam em `/kaggle/working/saida_tipo2` e são reaproveitados.

**Mande o arquivo `saida_tipo2/resultados_tipo2.xlsx`** (e, se quiser, as três figuras `fig_*.png`).

In [ ]:
%%writefile redes.py
"""
redes.py — dados do HAM10000, split por lesão, várias arquiteturas com a MESMA cabeça, treino e limiar.

Tudo o que muda de uma arquitetura para outra fica aqui:
  - como montar a rede (backbone pré-treinado no ImageNet + cabeça igual à do BRACIS);
  - quais são os PONTOS DE LEITURA (a saída de cada bloco), onde as probes vão ser colocadas.
O resto do código (leitura, fuzzy, avaliação) não sabe qual rede está olhando.
"""

import math
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from torchvision import models

MALIGNAS = {"mel", "bcc", "akiec"}
MEDIA = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
DESVIO = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
LADO_REDE = 224

# taxa de aprendizado do backbone por arquitetura (a cabeça usa 10x mais)
LR_BACKBONE = {"efficientnet_b0": 3e-4, "resnet50": 1e-4, "vit_b_16": 3e-5, "convnext_tiny": 1e-4}


# ============================================================================ dados
def carregar_ham(metadata_csv, img_dirs):
    meta = pd.read_csv(metadata_csv)
    meta["is_malignant"] = meta.dx.isin(MALIGNAS).astype(int)
    caminho = {p.stem: str(p) for d in img_dirs for p in Path(d).glob("*.jpg")}
    meta["image_path"] = meta.image_id.map(caminho)
    falta = meta.image_path.isna().sum()
    if falta:
        print(f"aviso: {falta} imagens do CSV não foram encontradas nas pastas")
    return meta.dropna(subset=["image_path"]).reset_index(drop=True)


def split_por_lesao(meta, frac_val=0.10, frac_teste=0.25, seed=42):
    """Mesma lesão nunca aparece em dois conjuntos. Estratificado por maligno/benigno.
    Cria as colunas split ∈ {treino, val, teste} e metade ∈ {0, 1} (só no teste, também por lesão)."""
    n_partes = 20
    dobra = np.zeros(len(meta), int)
    sgkf = StratifiedGroupKFold(n_splits=n_partes, shuffle=True, random_state=seed)
    for k, (_, te) in enumerate(sgkf.split(meta, meta.is_malignant, meta.lesion_id)):
        dobra[te] = k
    n_te, n_val = round(frac_teste * n_partes), round(frac_val * n_partes)
    meta = meta.copy()
    meta["split"] = np.where(dobra < n_te, "teste", np.where(dobra < n_te + n_val, "val", "treino"))
    meta["metade"] = -1
    t = meta.index[meta.split == "teste"]
    sg2 = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=seed)
    for k, (_, te) in enumerate(sg2.split(meta.loc[t], meta.loc[t, "is_malignant"], meta.loc[t, "lesion_id"])):
        meta.loc[t[te], "metade"] = k
    return meta


def carregar_cache(caminhos, lado=256, n_threads=8):
    """Todas as imagens em memória, redimensionadas para lado×lado (uint8). ~2 GB para o HAM10000 inteiro."""
    def um(c):
        im = Image.open(c).convert("RGB").resize((lado, lado), Image.BILINEAR)
        return torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1)
    t0 = time.time()
    with ThreadPoolExecutor(n_threads) as ex:
        ims = list(ex.map(um, list(caminhos)))
    cache = torch.stack(ims)
    print(f"cache: {len(cache)} imagens {lado}x{lado} em {time.time() - t0:.0f}s ({cache.numel() / 1e9:.1f} GB)")
    return cache


def preparar(xb, device):
    """uint8 (N,3,H,W) → float normalizado 224×224 no device."""
    x = xb.to(device, non_blocking=True).float().div_(255)
    if x.shape[-1] != LADO_REDE:
        x = F.interpolate(x, size=(LADO_REDE, LADO_REDE), mode="bilinear", antialias=True, align_corners=False)
    return (x - MEDIA.to(device)) / DESVIO.to(device)


def aumentar(x):
    """Aumento de dados na GPU, diferente para cada imagem: giro qualquer (lesão não tem 'cima'),
    espelho, zoom 0.8–1.0, pequeno deslocamento, brilho e contraste. x já normalizado."""
    n, dev = x.shape[0], x.device
    ang = (torch.rand(n, device=dev) * 2 - 1) * math.pi
    esc = 0.8 + 0.2 * torch.rand(n, device=dev)
    esp = torch.where(torch.rand(n, device=dev) < 0.5, -1.0, 1.0)
    tx, ty = [(torch.rand(n, device=dev) * 2 - 1) * 0.08 for _ in range(2)]
    c, s = torch.cos(ang) * esc, torch.sin(ang) * esc
    theta = torch.stack([torch.stack([c * esp, -s, tx], 1), torch.stack([s * esp, c, ty], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    brilho = (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.4
    contraste = 1 + (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.4
    m = x.mean((1, 2, 3), keepdim=True)
    return (x - m) * contraste + m + brilho


def lotes(cache, idx, bs):
    idx = np.asarray(idx)
    for i in range(0, len(idx), bs):
        yield cache[torch.from_numpy(idx[i:i + bs])]


# ============================================================================ redes
class Rede(nn.Module):
    """Backbone (devolve um vetor por imagem) + a cabeça do BRACIS. Saída = logit z; P = sigmoid(z)."""

    def __init__(self, backbone, dim, pontos, nome):
        super().__init__()
        self.backbone, self.nome = backbone, nome
        self.cabeca = nn.Sequential(nn.Dropout(0.4), nn.Linear(dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1))
        self._pontos = pontos          # lista de módulos (não registrada de novo: já estão dentro do backbone)

    @property
    def pontos(self):
        return self._pontos

    def forward(self, x):
        return self.cabeca(self.backbone(x)).view(-1)


def _base(nome, pretreinado, pesos_locais):
    fab = getattr(models, nome)
    if pesos_locais:
        m = fab(weights=None)
        m.load_state_dict(torch.load(pesos_locais, map_location="cpu"))
        return m
    if not pretreinado:
        return fab(weights=None)
    try:
        return fab(weights="DEFAULT")
    except Exception as e:
        raise RuntimeError(
            f"Não consegui baixar os pesos do ImageNet para {nome} ({e}).\n"
            "No Kaggle: Settings → Internet → On. Sem internet: anexe os arquivos .pth dos pesos do torchvision "
            "como dataset e informe o caminho em PESOS_LOCAIS.") from e


def criar_rede(nome, pretreinado=True, pesos_locais=None):
    m = _base(nome, pretreinado, pesos_locais)
    if nome == "efficientnet_b0":
        dim = m.classifier[-1].in_features; m.classifier = nn.Identity()
        pontos = [m.features[0]] + [b for est in list(m.features.children())[1:-1] for b in est.children()] + [m.features[-1]]
    elif nome == "resnet50":
        dim = m.fc.in_features; m.fc = nn.Identity()
        pontos = [m.maxpool] + [b for camada in (m.layer1, m.layer2, m.layer3, m.layer4) for b in camada.children()]
    elif nome == "vit_b_16":
        dim = m.heads.head.in_features; m.heads = nn.Identity()
        pontos = list(m.encoder.layers.children())
    elif nome == "convnext_tiny":
        dim = m.classifier[-1].in_features; m.classifier[-1] = nn.Identity()
        pontos = [m.features[0]] + [b for i in (1, 3, 5, 7) for b in m.features[i].children()]
    else:
        raise ValueError(f"arquitetura não suportada: {nome}. Opções: efficientnet_b0, resnet50, vit_b_16, convnext_tiny")
    return Rede(m, dim, pontos, nome)


# ============================================================================ treino
@torch.no_grad()
def prever(rede, cache, idx, device, bs=64):
    rede.eval(); out = []
    for xb in lotes(cache, idx, bs):
        with torch.autocast(device_type="cuda", enabled=device == "cuda"):
            out.append(torch.sigmoid(rede(preparar(xb, device)).float()).cpu())
    return torch.cat(out).numpy()


def treinar(rede, cache, idx_tr, y_tr, idx_val, y_val, device, epocas=12, bs=32, lr_backbone=1e-4,
            paciencia=4, seed=42, log=print):
    """AdamW + OneCycle, precisão mista, aumento na GPU. Guarda o melhor estado pela AUROC na validação."""
    g = torch.Generator().manual_seed(seed)
    rede.to(device)
    y_tr = np.asarray(y_tr, np.float32)
    pos_w = torch.tensor(math.sqrt((y_tr == 0).sum() / max((y_tr == 1).sum(), 1)), device=device)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW([{"params": rede.backbone.parameters(), "lr": lr_backbone},
                             {"params": rede.cabeca.parameters(), "lr": lr_backbone * 10}], weight_decay=1e-4)
    passos = epocas * math.ceil(len(idx_tr) / bs)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr_backbone, lr_backbone * 10], total_steps=passos, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    idx_tr = np.asarray(idx_tr); hist, melhor, estado, sem_melhora = [], -1, None, 0
    for ep in range(epocas):
        rede.train(); t0 = time.time(); perm = torch.randperm(len(idx_tr), generator=g).numpy(); tot = 0.0
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            if len(b) < 2:
                continue
            x = aumentar(preparar(cache[torch.from_numpy(idx_tr[b])], device))
            y = torch.from_numpy(y_tr[b]).to(device)
            with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                perda = lossf(rede(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(perda).backward(); scaler.step(opt); scaler.update(); sch.step()
            tot += perda.item() * len(b)
        p_val = prever(rede, cache, idx_val, device)
        auc = roc_auc_score(y_val, p_val) if len(np.unique(y_val)) == 2 else float("nan")
        hist.append({"época": ep + 1, "perda treino": tot / len(perm), "AUROC val": auc, "segundos": time.time() - t0})
        log(f"  {rede.nome} época {ep + 1}/{epocas}: perda {tot / len(perm):.4f} | AUROC val {auc:.4f} | {time.time() - t0:.0f}s")
        if auc > melhor:
            melhor, sem_melhora = auc, 0
            estado = {k: v.detach().cpu().clone() for k, v in rede.state_dict().items()}
        else:
            sem_melhora += 1
            if sem_melhora >= paciencia:
                log(f"  parada antecipada (sem melhora há {paciencia} épocas)"); break
    if estado is not None:
        rede.load_state_dict(estado)
    return rede.eval(), pd.DataFrame(hist)


def escolher_limiar(p, y):
    """Limiar que maximiza a acurácia balanceada (Youden) na validação."""
    p, y = np.asarray(p), np.asarray(y)
    cands = np.unique(np.quantile(p, np.linspace(0.01, 0.99, 197)))
    ba = [0.5 * (((p >= t) & (y == 1)).sum() / max(y.sum(), 1) + ((p < t) & (y == 0)).sum() / max((1 - y).sum(), 1))
          for t in cands]
    return float(cands[int(np.argmax(ba))])

In [ ]:
%%writefile leitura_geral.py
"""
leitura_geral.py — lê o comportamento interno de QUALQUER rede (EfficientNet, ResNet, ViT, ConvNeXt).

O leitor não sabe o que cada bloco "enxerga". Ele só precisa:
  1. dos pontos de leitura (saída de cada bloco), vindos de redes.py;
  2. de reduzir cada saída a um vetor: média espacial (CNN) ou token de classe (ViT);
  3. de uma probe (regressão logística) por ponto, que diz o que a rede responderia se parasse ali.

Para as features valerem entre redes com números diferentes de blocos, a PROFUNDIDADE é medida de 0 a 1:
  taxa_trocas      trocas de opinião / (nº de blocos − 1)   (a contagem crua depende do nº de blocos)
  prof_decisao     fração da rede a partir da qual a opinião não muda mais
  entropia_probes  indecisão média das camadas
  conf_fim         concordância com a resposta final no último terço da rede
  salto_final      a cabeça discorda do último bloco
  vistas_mudam, p_vistas_std, trajetoria_vistas   estabilidade entre versões espelhadas/giradas
  vizinhos_discordam, dist_vizinhos               familiaridade (vizinhos no último ponto de leitura)
  prox_p           proximidade de P ao limiar (a referência a superar)
Também devolve a trajetória reamostrada em 10 profundidades relativas (traj_r0..traj_r9), igual para todas as redes.
"""

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from redes import lotes, preparar

EPS = 1e-9
FEATURES = ["taxa_trocas", "prof_decisao", "entropia_probes", "conf_fim", "salto_final",
            "vistas_mudam", "p_vistas_std", "trajetoria_vistas", "vizinhos_discordam", "dist_vizinhos", "prox_p"]
VISTAS = {"original": lambda x: x, "espelho_h": lambda x: torch.flip(x, [3]),
          "espelho_v": lambda x: torch.flip(x, [2]), "giro_180": lambda x: torch.flip(x, [2, 3])}
N_REL = 10


def reduzir(o):
    """Saída de um bloco → um vetor por imagem."""
    if isinstance(o, (tuple, list)):
        o = o[0]
    if o.dim() == 4:
        return o.float().mean((2, 3))      # CNN: média espacial
    if o.dim() == 3:
        return o[:, 0].float()             # ViT: token de classe
    return o.flatten(1).float()


class LeitorGeral:
    def __init__(self, rede, limiar=0.5, k_vizinhos=15, bs=64):
        self.rede, self.limiar, self.k, self.bs = rede, limiar, k_vizinhos, bs
        self.pontos = list(rede.pontos)
        self.B = len(self.pontos)

    @torch.no_grad()
    def _ler(self, cache, idx, device, vista=lambda x: x):
        self.rede.eval()
        acum, P = [[] for _ in range(self.B)], []
        for xb in lotes(cache, idx, self.bs):
            cap = [None] * self.B
            hs = [b.register_forward_hook(lambda m, i, o, k=k: cap.__setitem__(k, reduzir(o).cpu().numpy()))
                  for k, b in enumerate(self.pontos)]
            try:
                with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                    z = self.rede(vista(preparar(xb, device)))
                P.append(torch.sigmoid(z.float()).cpu().numpy())
            finally:
                for h in hs:
                    h.remove()
            for k in range(self.B):
                acum[k].append(cap[k])
        return [np.concatenate(a).astype(np.float32) for a in acum], np.concatenate(P)

    def fit(self, cache, idx_ref, y_ref, device):
        """Probes e banco de vizinhos a partir de imagens que a rede viu no treino."""
        A, P = self._ler(cache, idx_ref, device)
        dec = (P >= self.limiar).astype(int)
        alvo = dec if len(np.unique(dec)) == 2 else np.asarray(y_ref).astype(int)
        self.probes = [make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000, class_weight="balanced"))
                       .fit(a, alvo) for a in A]
        self.banco, self.banco_dec = self._unit(A[-1]), dec
        self.dist_ref = float(np.median(self._knn_dist(self.banco, excluir_si=True))) + EPS
        return self

    @staticmethod
    def _unit(a):
        a = a - a.mean(1, keepdims=True)
        return a / (np.linalg.norm(a, axis=1, keepdims=True) + EPS)

    def _knn(self, U, excluir_si=False):
        S = U @ self.banco.T
        if excluir_si:
            np.fill_diagonal(S, -np.inf)
        k = min(self.k, S.shape[1] - (1 if excluir_si else 0))
        return np.argsort(-S, 1)[:, :k], S

    def _knn_dist(self, U, excluir_si=False):
        idx, S = self._knn(U, excluir_si)
        return (1 - np.take_along_axis(S, idx, 1)).mean(1)

    def _trajetoria(self, A):
        return np.stack([pr.predict_proba(a)[:, 1] for pr, a in zip(self.probes, A)], 1)   # (n, B)

    def transform(self, cache, idx, device):
        res = {nome: self._ler(cache, idx, device, v) for nome, v in VISTAS.items()}
        A, P = res["original"]
        pp = self._trajetoria(A)
        B = self.B
        final = (P >= self.limiar).astype(int)
        f = {"p": P}
        op = (pp >= 0.5).astype(int)
        trocas = np.abs(np.diff(op, axis=1)).sum(1)
        f["trocas_opiniao"] = trocas                                   # contagem crua (só para consulta)
        f["taxa_trocas"] = trocas / max(B - 1, 1)
        f["prof_decisao"] = np.where(op != final[:, None], np.arange(B)[None, :] + 1, 0).max(1) / B
        f["entropia_probes"] = -(pp * np.log(pp + EPS) + (1 - pp) * np.log(1 - pp + EPS)).mean(1)
        conf = np.where(final[:, None] == 1, pp, 1 - pp)
        f["conf_fim"] = conf[:, -int(np.ceil(B / 3)):].mean(1)
        f["salto_final"] = np.abs(P - pp[:, -1])
        Ps = np.stack([res[v][1] for v in VISTAS], 1)
        f["vistas_mudam"] = ((Ps[:, 1:] >= self.limiar).astype(int) != final[:, None]).mean(1)
        f["p_vistas_std"] = Ps.std(1)
        T = np.stack([self._trajetoria(res[v][0]) for v in VISTAS], 0)
        f["trajetoria_vistas"] = T.std(0).mean(1)
        idx_v, S = self._knn(self._unit(A[-1]))
        f["vizinhos_discordam"] = (self.banco_dec[idx_v] != final[:, None]).mean(1)
        f["dist_vizinhos"] = (1 - np.take_along_axis(S, idx_v, 1)).mean(1) / self.dist_ref
        f["prox_p"] = 1 - np.abs(P - self.limiar) / max(self.limiar, 1 - self.limiar)
        # trajetória em profundidade relativa (0 = primeiro bloco, 1 = último), comparável entre redes
        x_rel, grade = np.linspace(0, 1, B), np.linspace(0, 1, N_REL)
        conf_rel = np.stack([np.interp(grade, x_rel, c) for c in conf])
        for j in range(N_REL):
            f[f"traj_r{j}"] = conf_rel[:, j]
        return pd.DataFrame(f)


def ler(leitor, meta, idx, cache, device):
    """Uma linha por imagem do conjunto idx: features + y_true, y_pred, erro (+ split/metade/lesão)."""
    t = leitor.transform(cache, idx, device)
    m = meta.iloc[idx].reset_index(drop=True)
    for c in ["image_id", "lesion_id", "metade"]:
        t.insert(0, c, m[c].values)
    t["y_true"] = m.is_malignant.values.astype(int)
    t["y_pred"] = (t.p >= leitor.limiar).astype(int)
    t["erro"] = (t.y_pred != t.y_true).astype(int)
    return t

In [ ]:
%%writefile fuzzy_t2.py
"""
fuzzy_t2.py — Wang-Mendel tipo 1 e tipo 2 intervalar, e o teste "treina em umas redes, aplica em outra".

TIPO 1: cada feature tem 3 termos (BAIXA, MÉDIA, ALTA), triângulos com picos nos quantis 5%, 50%, 95%.

TIPO 2 (envelope entre redes): cada rede de origem tem os seus próprios picos. Para cada termo,
  pertinência inferior = a menor entre as redes, superior = a maior.
  A faixa entre as duas (a "pegada de incerteza") é exatamente a discordância entre as redes sobre
  o que é, por exemplo, "taxa_trocas ALTA".

INFERÊNCIA DO TIPO 2: cada regra dispara com uma força num intervalo [f_inf, f_sup]
  (produto das pertinências inferiores / superiores). A nota de erro é
      s = E / (E + A),   E = Σ força·CF das regras ERRO,   A = Σ força·CF das regras ACERTO.
  s cresce com E e decresce com A, então os extremos do intervalo saem direto (sem iteração de Karnik-Mendel):
      s_inf = E_inf / (E_inf + A_sup),   s_sup = E_sup / (E_sup + A_inf).
  Nota pontual = a nota com as pertinências do MEIO da faixa (fica sempre dentro do intervalo).
  Largura = o quanto as redes discordam naquele caso; [0, 1] = discordância total (termos sem sobreposição).

TRÊS ZONAS: limiar τ escolhido nas redes de origem. alarme: s_inf > τ · seguro: s_sup < τ · incerto: o resto.
"""

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

EPS = 1e-12
TERMOS = ["BAIXA", "MÉDIA", "ALTA"]


# ============================================================================ pertinências
def centros(X):
    """Picos dos 3 termos por feature (J, 3): quantis 5%, 50%, 95%, estritamente crescentes."""
    C = np.nanquantile(np.asarray(X, float), [0.05, 0.5, 0.95], axis=0).T.copy()
    for j in range(len(C)):
        for k in (1, 2):
            C[j, k] = max(C[j, k], C[j, k - 1] + 1e-6 * (abs(C[j, k - 1]) + 1))
    return C


def pert_t1(X, C):
    """Pertinências (n, J, 3) com triângulos que somam 1."""
    X = np.asarray(X, float); n, J = X.shape
    M = np.zeros((n, J, 3))
    for j in range(J):
        c = C[j]; x = np.nan_to_num(X[:, j], nan=c[1])
        M[:, j, 0] = x <= c[0]; M[:, j, 2] = x >= c[2]
        for k in (0, 1):
            d = (x >= c[k]) & (x < c[k + 1]); t = (x[d] - c[k]) / (c[k + 1] - c[k])
            M[d, j, k] = 1 - t; M[d, j, k + 1] = t
    return M


def pert_t2(X, lista_C):
    """Envelope das pertinências tipo 1 de várias redes: (inferior, superior), cada um (n, J, 3)."""
    Ms = np.stack([pert_t1(X, C) for C in lista_C])
    return Ms.min(0), Ms.max(0)


# ============================================================================ Wang-Mendel
class WangMendel:
    """Regras aprendidas a partir de pertinências já calculadas (tipo 1, ou o ponto médio do tipo 2).
    Cada exemplo propõe a regra com o termo de maior grau em cada feature; ERRO/ACERTO e o fator de certeza (CF)
    vêm de todos os exemplos compatíveis. Pesos: classes balanceadas DENTRO de cada rede de origem e
    cada rede com o mesmo peso total (uma rede não domina só por ter mais erros)."""

    def __init__(self, min_suporte=0.01, min_cf=0.1):
        self.min_suporte, self.min_cf = min_suporte, min_cf

    def fit(self, M, y, nomes, grupos=None):
        y = np.asarray(y).astype(int); n = len(y)
        grupos = np.zeros(n, int) if grupos is None else np.asarray(grupos)
        self.nomes = list(nomes)
        w = np.zeros(n); G = np.unique(grupos)
        for g in G:
            m = grupos == g
            for c in (0, 1):
                mc = m & (y == c)
                if mc.any():
                    w[mc] = 1.0 / (2 * mc.sum() * len(G))
        ant_por_ex = M.argmax(2)                                    # (n, J)
        self.regras_ = []
        for ant in {tuple(a) for a in ant_por_ex}:
            comp = self._forca_uma(M, ant)
            sup = comp.sum() / n
            b = np.array([(comp * w)[y == 0].sum(), (comp * w)[y == 1].sum()])
            if sup < self.min_suporte or b.sum() <= EPS:
                continue
            c = int(b.argmax()); cf = (b[c] - b[1 - c]) / b.sum()
            if cf >= self.min_cf:
                self.regras_.append({"ant": ant, "cons": c, "cf": float(cf), "suporte": float(sup)})
        self.regras_.sort(key=lambda r: -r["suporte"] * r["cf"])
        return self

    @staticmethod
    def _forca_uma(M, ant):
        return np.prod(M[:, np.arange(len(ant)), list(ant)], axis=1)

    def _somas(self, M):
        E = np.zeros(len(M)); A = np.zeros(len(M))
        for r in self.regras_:
            f = self._forca_uma(M, r["ant"]) * r["cf"]
            if r["cons"] == 1:
                E += f
            else:
                A += f
        return E, A

    def nota_t1(self, M):
        E, A = self._somas(M)
        return np.where(E + A > EPS, E / np.maximum(E + A, EPS), 0.5)

    def nota_t2(self, M_inf, M_sup):
        """(s_inf, s_sup, s_meio) para pertinências intervalares. s_meio usa as pertinências do meio da faixa,
        as mesmas com que as regras foram aprendidas."""
        E_i, A_i = self._somas(M_inf)
        E_s, A_s = self._somas(M_sup)
        s_inf = np.where(E_i + A_s > EPS, E_i / np.maximum(E_i + A_s, EPS), 0.5)
        s_sup = np.where(E_s + A_i > EPS, E_s / np.maximum(E_s + A_i, EPS), 0.5)
        s_sup = np.maximum(s_sup, s_inf)
        s_meio = np.clip(self.nota_t1((M_inf + M_sup) / 2), s_inf, s_sup)
        return s_inf, s_sup, s_meio

    @property
    def n_regras(self):
        return len(self.regras_)

    def tabela(self):
        return pd.DataFrame([{"regra": "SE " + " E ".join(f"{self.nomes[j]} é {TERMOS[k]}" for j, k in enumerate(r["ant"]))
                              + (" ENTÃO ERRO" if r["cons"] else " ENTÃO ACERTO"),
                              "certeza (CF)": round(r["cf"], 3), "suporte": round(r["suporte"], 3)} for r in self.regras_])


def tabela_fou(lista_C, nomes, rotulos):
    """Onde fica o pico de cada termo em cada rede, e a faixa (mín–máx) que vira a incerteza do tipo 2."""
    lin = []
    Cs = np.stack(lista_C)                                          # (S, J, 3)
    for j, f in enumerate(nomes):
        for k, t in enumerate(TERMOS):
            d = {"feature": f, "termo": t}
            for s, r in enumerate(rotulos):
                d[f"pico em {r}"] = Cs[s, j, k]
            d["faixa (mín)"], d["faixa (máx)"] = Cs[:, j, k].min(), Cs[:, j, k].max()
            d["largura relativa"] = (Cs[:, j, k].max() - Cs[:, j, k].min()) / (abs(Cs[:, j, 2] - Cs[:, j, 0]).mean() + EPS)
            lin.append(d)
    return pd.DataFrame(lin)


# ============================================================================ seleção e métricas
def selecionar(tab, cand, k=3, forcar=("prox_p",), max_corr=0.85):
    """Top-k por AUROC (para detectar o erro), sem features muito correlacionadas. `forcar` entram sempre."""
    def forca(f):
        return abs(roc_auc_score(tab.erro, tab[f]) - 0.5) if tab[f].nunique() > 1 and tab.erro.nunique() > 1 else 0
    esc = [f for f in forcar if f in cand]
    for f in sorted(cand, key=lambda f: -forca(f)):
        if len(esc) >= k:
            break
        if f not in esc and all(abs(spearmanr(tab[f], tab[g]).correlation) < max_corr for g in esc):
            esc.append(f)
    return esc


def auroc_alem_de_p(y, s, prox_p, faixas=5):
    """AUROC dentro de faixas de P parecido (média ponderada): o que a nota traz além de P."""
    y, s = np.asarray(y), np.asarray(s)
    bins = pd.qcut(pd.Series(prox_p).rank(method="first"), faixas, labels=False).values
    vals, pesos = [], []
    for b in range(faixas):
        m = bins == b
        if m.any() and y[m].min() != y[m].max():
            vals.append(roc_auc_score(y[m], s[m])); pesos.append(m.sum())
    return float(np.average(vals, weights=pesos)) if vals else np.nan


def aurc(y, s):
    """Área sob a curva risco-cobertura: aceita primeiro as imagens com menor nota de erro. Menor = melhor."""
    o = np.argsort(s, kind="stable"); e = np.asarray(y)[o]
    return float((np.cumsum(e) / np.arange(1, len(e) + 1)).mean())


def metricas(y, s, prox_p):
    if len(np.unique(y)) < 2:
        return {"AUROC": np.nan, "AUROC além de P": np.nan, "AUPR do erro": np.nan, "AURC (menor=melhor)": np.nan}
    return {"AUROC": roc_auc_score(y, s), "AUROC além de P": auroc_alem_de_p(y, s, prox_p),
            "AUPR do erro": average_precision_score(y, s), "AURC (menor=melhor)": aurc(y, s)}


def ic_diferenca(y, s1, s2, n_boot=1000, seed=0):
    """IC 95% bootstrap (pareado) da diferença de AUROC s1 − s2."""
    r = np.random.default_rng(seed); y = np.asarray(y); d = []
    if len(np.unique(y)) < 2:
        return np.nan, np.nan
    for _ in range(n_boot):
        i = r.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            d.append(roc_auc_score(y[i], s1[i]) - roc_auc_score(y[i], s2[i]))
    return (np.percentile(d, 2.5), np.percentile(d, 97.5)) if d else (np.nan, np.nan)


# ============================================================================ fuzzy aditivo
class FuzzyAditivo:
    """Regras de UMA condição cada, somadas: 'SE taxa_trocas é ALTA → +1,3 no risco de erro (log-odds)'.
    Pesos aprendidos por regressão logística sobre os graus de pertinência (mesmo balanceamento do Wang-Mendel).
    No tipo 2 a nota é linear nas pertinências, então o intervalo sai exato:
      z_inf = b + Σ (w>0 ? w·μ_inf : w·μ_sup),   z_sup = b + Σ (w>0 ? w·μ_sup : w·μ_inf)."""

    def __init__(self, C=1.0):
        self.C = C

    def fit(self, M, y, nomes, grupos=None):
        y = np.asarray(y).astype(int); n = len(y); self.nomes = list(nomes)
        grupos = np.zeros(n, int) if grupos is None else np.asarray(grupos)
        w = np.ones(n); G = np.unique(grupos)
        for g in G:
            for c in (0, 1):
                mc = (grupos == g) & (y == c)
                if mc.any():
                    w[mc] = n / (2 * mc.sum() * len(G))
        self.ok = len(np.unique(y)) == 2
        if self.ok:
            self.lr = LogisticRegression(C=self.C, max_iter=3000).fit(M.reshape(n, -1), y, sample_weight=w)
            self.w, self.b = self.lr.coef_[0], self.lr.intercept_[0]
        return self

    def nota_t1(self, M):
        if not self.ok:
            return np.full(len(M), 0.5)
        return 1 / (1 + np.exp(-(M.reshape(len(M), -1) @ self.w + self.b)))

    def nota_t2(self, M_inf, M_sup):
        if not self.ok:
            h = np.full(len(M_inf), 0.5); return h, h, h
        a, s = M_inf.reshape(len(M_inf), -1), M_sup.reshape(len(M_sup), -1)
        pos = self.w > 0
        z_i = a[:, pos] @ self.w[pos] + s[:, ~pos] @ self.w[~pos] + self.b
        z_s = s[:, pos] @ self.w[pos] + a[:, ~pos] @ self.w[~pos] + self.b
        sig = lambda z: 1 / (1 + np.exp(-z))
        s_i, s_s = sig(z_i), sig(z_s)
        return s_i, s_s, np.clip(self.nota_t1((M_inf + M_sup) / 2), s_i, s_s)

    def tabela(self):
        if not self.ok:
            return pd.DataFrame()
        lin = [{"regra": f"SE {f} é {TERMOS[k]}", "peso no risco de erro": round(float(self.w[3 * j + k]), 3),
                "efeito": "aumenta" if self.w[3 * j + k] > 0.05 else ("diminui" if self.w[3 * j + k] < -0.05 else "neutro")}
               for j, f in enumerate(self.nomes) for k in range(3)]
        lin.append({"regra": "(base)", "peso no risco de erro": round(float(self.b), 3), "efeito": ""})
        return pd.DataFrame(lin)


# ============================================================================ o experimento
REFERENCIAS = ["Só P", "Regressão logística dentro (referência)", "Regressão logística transferida", "Boosting transferido"]
VARIANTES = ["tipo 1 dentro (referência)", "tipo 1 transferido", "tipo 1 transferido + termos no alvo",
             "tipo 2 transferido", "tipo 2 transferido + alvo no envelope"]
FAMILIAS = ["Wang-Mendel", "Fuzzy aditivo"]
METODOS = REFERENCIAS + [f"{fam}: {v}" for fam in FAMILIAS for v in VARIANTES]
PRINCIPAIS = REFERENCIAS + [f"Fuzzy aditivo: {v}" for v in VARIANTES]      # o que vai para o artigo
INTERNOS = [f"Wang-Mendel: {v}" for v in VARIANTES]                        # só para consulta (fora do artigo)


def _pesos(y, g):
    w = np.ones(len(y)); G = np.unique(g)
    for gg in G:
        for c in (0, 1):
            m = (g == gg) & (y == c)
            if m.any():
                w[m] = len(y) / (2 * m.sum() * len(G))
    return w


def _zonas(alvo, sistema, y, s_inf, s_sup, tau):
    out = []
    for nome_z, mz in [("alarme", s_inf > tau), ("incerto", (s_inf <= tau) & (s_sup >= tau)), ("seguro", s_sup < tau)]:
        if nome_z == "incerto" and np.allclose(s_inf, s_sup, equal_nan=True):
            continue
        out.append({"rede alvo": alvo, "sistema": sistema, "zona": nome_z, "imagens": int(mz.sum()),
                    "% das imagens": mz.mean(), "taxa de erro da rede na zona": y[mz].mean() if mz.any() else np.nan,
                    "erros na zona": int(y[mz].sum()), "% dos erros": y[mz].sum() / max(y.sum(), 1)})
    return out


def deixa_uma_fora(tabs, cand, k_wm=3, k_ad=5, forcar=("prox_p",), q_alarme=0.2, n_boot=1000, log=print):
    """Para cada rede ALVO: os sistemas são aprendidos nas OUTRAS redes e aplicados no alvo.
    O teste de cada rede é dividido em duas metades por lesão: aprende numa metade e avalia na outra (e troca),
    então nenhuma imagem é usada para aprender e avaliar ao mesmo tempo.
    Duas famílias fuzzy, cada uma em tipo 1 e tipo 2: Wang-Mendel (regras com k_wm condições) e fuzzy aditivo
    (regras de uma condição somadas, k_ad features, o mesmo conjunto da regressão logística e do boosting).
    'Termos no alvo' e 'alvo no envelope' usam só as features do alvo, nunca os rótulos dele."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    nomes = list(tabs)
    linhas, zonas, regras, fous, preds = [], [], {}, {}, {}
    for alvo in nomes:
        fontes = [s for s in nomes if s != alvo]
        T = tabs[alvo]; n = len(T)
        nota = {m: np.full(n, np.nan) for m in METODOS}
        iv = {fam: [np.full(n, np.nan) for _ in range(3)] for fam in FAMILIAS}      # s_inf, s_sup, tau
        escolhas = []
        for f in (0, 1):
            te = (T.metade == 1 - f).values
            fonte_tr = pd.concat([tabs[s][tabs[s].metade == f].assign(_g=i) for i, s in enumerate(fontes)], ignore_index=True)
            alvo_tr = T[T.metade == f]
            ys, gs = fonte_tr.erro.values, fonte_tr._g.values
            nota["Só P"][te] = T.loc[te, "prox_p"].values
            f_ad = selecionar(fonte_tr, cand, k_ad, forcar); f_wm = selecionar(fonte_tr, cand, k_wm, forcar)
            escolhas.append((f_wm, f_ad))
            # referências não fuzzy (k_ad features)
            Xs, Xa, Xin = fonte_tr[f_ad].values, T.loc[te, f_ad].values, alvo_tr[f_ad].values
            def lr_fit(X, y, w=None):
                return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(
                    X, y, logisticregression__sample_weight=w) if len(np.unique(y)) == 2 else None
            prob = lambda mdl, X: mdl.predict_proba(X)[:, 1] if mdl is not None else np.full(len(X), 0.5)
            nota["Regressão logística dentro (referência)"][te] = prob(lr_fit(Xin, alvo_tr.erro.values, _pesos(alvo_tr.erro.values, np.zeros(len(alvo_tr)))), Xa)
            nota["Regressão logística transferida"][te] = prob(lr_fit(Xs, ys, _pesos(ys, gs)), Xa)
            if len(np.unique(ys)) == 2:
                gb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=0)
                nota["Boosting transferido"][te] = gb.fit(Xs, ys, sample_weight=_pesos(ys, gs)).predict_proba(Xa)[:, 1]
            else:
                nota["Boosting transferido"][te] = 0.5
            # as duas famílias fuzzy
            for fam, feats, Cls in [("Wang-Mendel", f_wm, WangMendel), ("Fuzzy aditivo", f_ad, FuzzyAditivo)]:
                Xs, Xa, Xin = fonte_tr[feats].values, T.loc[te, feats].values, alvo_tr[feats].values
                Ca = centros(Xin)
                nota[f"{fam}: tipo 1 dentro (referência)"][te] = Cls().fit(pert_t1(Xin, Ca), alvo_tr.erro.values, feats).nota_t1(pert_t1(Xa, Ca))
                C1 = centros(Xs)
                m1 = Cls().fit(pert_t1(Xs, C1), ys, feats, gs)
                nota[f"{fam}: tipo 1 transferido"][te] = m1.nota_t1(pert_t1(Xa, C1))
                nota[f"{fam}: tipo 1 transferido + termos no alvo"][te] = m1.nota_t1(pert_t1(Xa, centros(Xa)))
                lista_C = [centros(fonte_tr[fonte_tr._g == i][feats].values) for i in range(len(fontes))]
                Mi, Ms = pert_t2(Xs, lista_C)
                m2 = Cls().fit((Mi + Ms) / 2, ys, feats, gs)
                s_i, s_s, s_m = m2.nota_t2(*pert_t2(Xa, lista_C))
                nota[f"{fam}: tipo 2 transferido"][te] = s_m
                nota[f"{fam}: tipo 2 transferido + alvo no envelope"][te] = m2.nota_t2(*pert_t2(Xa, lista_C + [centros(Xa)]))[2]
                iv[fam][0][te], iv[fam][1][te] = s_i, s_s
                iv[fam][2][te] = np.quantile(m2.nota_t2(Mi, Ms)[2], 1 - q_alarme)      # limiar escolhido nas FONTES
                if f == 0:
                    regras[(alvo, fam)] = m2.tabela()
                    fous[(alvo, fam)] = tabela_fou(lista_C, feats, fontes)
        y, pp = T.erro.values, T.prox_p.values
        auc = lambda s: roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan
        for m in METODOS:
            d = {"rede alvo": alvo, "aprendido em": " + ".join(fontes), "método": m, **metricas(y, nota[m], pp)}
            if ": tipo 2" in m:
                ref = nota[m.split(":")[0] + ": tipo 1 transferido"]
                lo, hi = ic_diferenca(y, nota[m], ref, n_boot)
                d["ganho de AUROC sobre o tipo 1 transferido"] = auc(nota[m]) - auc(ref)
                d["IC 95% do ganho"] = f"[{lo:+.3f}, {hi:+.3f}]"
            linhas.append(d)
        for fam in FAMILIAS:
            zonas += _zonas(alvo, f"{fam} tipo 2", y, *iv[fam])
        p = T[["image_id", "metade", "erro", "p", "prox_p"]].copy()
        for m in METODOS:
            p[m] = nota[m]
        for fam in FAMILIAS:
            p[f"{fam} t2 inf"], p[f"{fam} t2 sup"] = iv[fam][0], iv[fam][1]
            p[f"{fam} t2 largura"] = iv[fam][1] - iv[fam][0]
        preds[alvo] = p
        log(f"alvo {alvo}: WM {escolhas[0][0]} | aditivo/LR {escolhas[0][1]}\n   " + " | ".join(
            f"{m.replace('transferido', 'transf.')}: {auc(nota[m]):.3f}" for m in
            ["Só P", "Regressão logística transferida", "Wang-Mendel: tipo 1 transferido", "Wang-Mendel: tipo 2 transferido",
             "Fuzzy aditivo: tipo 1 transferido", "Fuzzy aditivo: tipo 2 transferido"]))
    return pd.DataFrame(linhas), pd.DataFrame(zonas), regras, fous, preds

In [ ]:
%%writefile experimento.py
"""
experimento.py — o pipeline inteiro, usado pelo notebook e pelo rodar.py.

  1. dados: HAM10000, split por lesão (treino / val / teste; o teste em duas metades)
  2. para cada arquitetura: treina (ou carrega o checkpoint), escolhe o limiar na validação
  3. para cada arquitetura: lê a rede por dentro no teste (features em profundidade relativa)
  4. deixa-uma-arquitetura-fora: tipo 1 × tipo 2
  5. salva planilha + figuras

Tudo que é caro fica salvo em cfg.saida (checkpoints, limiares, features). Se o notebook cair no meio,
rodar de novo continua de onde parou.
"""

import json
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import fuzzy_t2 as ft
import leitura_geral as lg
import redes as rd


@dataclass
class Config:
    metadata_csv: str = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"
    img_dirs: list = field(default_factory=lambda: ["/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1",
                                                     "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2"])
    saida: str = "saida_tipo2"
    arquiteturas: list = field(default_factory=lambda: ["efficientnet_b0", "resnet50", "vit_b_16"])
    epocas: int = 15
    bs: int = 32
    paciencia: int = 4
    lado_cache: int = 256
    frac_val: float = 0.10
    frac_teste: float = 0.25
    n_ref: int = 2000                 # imagens do treino usadas para treinar as probes e o banco de vizinhos
    k_features: int = 3               # features por regra (Wang-Mendel)
    k_aditivo: int = 5                # features do fuzzy aditivo, da regressão logística e do boosting
    forcar: tuple = ("prox_p",)       # features que sempre entram (P é a pista mais forte)
    q_alarme: float = 0.20            # fração das imagens de origem acima do limiar do alarme
    n_boot: int = 1000
    pretreinado: bool = True
    pesos_locais: dict = field(default_factory=dict)   # {"resnet50": "/kaggle/input/.../resnet50.pth"} se não houver internet
    lr_backbone: dict = field(default_factory=dict)    # sobrescreve redes.LR_BACKBONE
    max_imagens: int = 0              # >0: usa só uma amostra (teste rápido)
    seed: int = 42


def _dev():
    return "cuda" if torch.cuda.is_available() else "cpu"


def preparar_dados(cfg, log=print):
    meta = rd.carregar_ham(cfg.metadata_csv, cfg.img_dirs)
    if cfg.max_imagens and len(meta) > cfg.max_imagens:
        lesoes = meta.lesion_id.drop_duplicates().sample(frac=1, random_state=cfg.seed)
        meta = meta[meta.lesion_id.isin(set(lesoes.iloc[:cfg.max_imagens]))].reset_index(drop=True)
    meta = rd.split_por_lesao(meta, cfg.frac_val, cfg.frac_teste, cfg.seed)
    out = Path(cfg.saida); out.mkdir(parents=True, exist_ok=True)
    arq = out / "split.csv"
    if arq.exists():                                   # garante o MESMO split numa segunda execução
        antigo = pd.read_csv(arq)
        meta = meta.drop(columns=["split", "metade"]).merge(antigo[["image_id", "split", "metade"]], on="image_id")
        log("split: reaproveitado de split.csv")
    else:
        meta[["image_id", "lesion_id", "dx", "is_malignant", "split", "metade"]].to_csv(arq, index=False)
    resumo = meta.groupby("split").agg(imagens=("image_id", "size"), lesoes=("lesion_id", "nunique"),
                                       malignas=("is_malignant", "mean"))
    log(resumo.round(3).to_string())
    cache = rd.carregar_cache(meta.image_path, cfg.lado_cache)
    return meta, cache


def treinar_ou_carregar(cfg, nome, meta, cache, log=print):
    out = Path(cfg.saida); ck, lj = out / f"rede_{nome}.pt", out / f"limiar_{nome}.json"
    dev = _dev()
    rede = rd.criar_rede(nome, cfg.pretreinado and not ck.exists(), cfg.pesos_locais.get(nome))
    idx = {s: np.where(meta.split == s)[0] for s in ("treino", "val", "teste")}
    y = meta.is_malignant.values
    if ck.exists() and lj.exists():
        rede.load_state_dict(torch.load(ck, map_location="cpu")); rede.to(dev).eval()
        info = json.load(open(lj)); log(f"{nome}: checkpoint carregado (limiar {info['limiar']:.3f})")
        hist = pd.read_csv(out / f"historico_{nome}.csv") if (out / f"historico_{nome}.csv").exists() else pd.DataFrame()
        return rede, info, hist
    lr = {**rd.LR_BACKBONE, **cfg.lr_backbone}.get(nome, 1e-4)
    log(f"{nome}: treinando ({len(idx['treino'])} imagens, {cfg.epocas} épocas, lr backbone {lr:g}, {dev})")
    torch.manual_seed(cfg.seed)
    rede, hist = rd.treinar(rede, cache, idx["treino"], y[idx["treino"]], idx["val"], y[idx["val"]], dev,
                            cfg.epocas, cfg.bs, lr, cfg.paciencia, cfg.seed, log)
    p_val = rd.prever(rede, cache, idx["val"], dev)
    lim = rd.escolher_limiar(p_val, y[idx["val"]])
    p_te = rd.prever(rede, cache, idx["teste"], dev)
    from sklearn.metrics import balanced_accuracy_score, roc_auc_score
    info = {"rede": nome, "limiar": lim, "AUROC val": float(roc_auc_score(y[idx["val"]], p_val)),
            "AUROC teste": float(roc_auc_score(y[idx["teste"]], p_te)),
            "acurácia balanceada teste": float(balanced_accuracy_score(y[idx["teste"]], p_te >= lim)),
            "taxa de erro teste": float(((p_te >= lim) != y[idx["teste"]]).mean()),
            "pontos de leitura": len(rede.pontos), "parâmetros (M)": sum(p.numel() for p in rede.parameters()) / 1e6}
    torch.save(rede.state_dict(), ck); json.dump(info, open(lj, "w"), indent=1); hist.to_csv(out / f"historico_{nome}.csv", index=False)
    log(f"{nome}: AUROC teste {info['AUROC teste']:.3f} | erro {info['taxa de erro teste']:.1%} | limiar {lim:.3f}")
    return rede, info, hist


def ler_ou_carregar(cfg, nome, rede, info, meta, cache, log=print):
    arq = Path(cfg.saida) / f"features_{nome}.csv"
    if arq.exists():
        log(f"{nome}: features carregadas de {arq.name}")
        return pd.read_csv(arq)
    dev = _dev(); t0 = time.time()
    tr = np.where(meta.split == "treino")[0]
    ref = np.random.default_rng(cfg.seed).choice(tr, size=min(cfg.n_ref, len(tr)), replace=False)
    leitor = lg.LeitorGeral(rede, info["limiar"]).fit(cache, ref, meta.is_malignant.values[ref], dev)
    tab = lg.ler(leitor, meta, np.where(meta.split == "teste")[0], cache, dev)
    tab.to_csv(arq, index=False)
    log(f"{nome}: {leitor.B} pontos de leitura | {tab.erro.sum()} erros em {len(tab)} imagens de teste | {time.time() - t0:.0f}s")
    return tab


def figuras(tabs, fous, preds, pasta, log=print):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    pasta = Path(pasta); arqs = []
    # 1) trajetória média em profundidade relativa, por rede: acertos × erros
    fig, axes = plt.subplots(1, len(tabs), figsize=(4.2 * len(tabs), 3.4), sharey=True)
    axes = np.atleast_1d(axes)
    cols = [f"traj_r{j}" for j in range(lg.N_REL)]; x = np.linspace(0, 1, lg.N_REL)
    for ax, (nome, t) in zip(axes, tabs.items()):
        for rot, m, cor in [("acertos", t.erro == 0, "#2a78d6"), ("erros", t.erro == 1, "#c0392b")]:
            if m.sum() == 0:
                continue
            q = np.percentile(t.loc[m, cols].values, [25, 50, 75], axis=0)
            ax.plot(x, q[1], color=cor, lw=2, label=f"{rot} (n={m.sum()})"); ax.fill_between(x, q[0], q[2], color=cor, alpha=.15)
        ax.axhline(.5, color="gray", ls="--", lw=1); ax.set_title(nome, fontsize=10); ax.set_xlabel("profundidade relativa (0 = início, 1 = fim)")
        ax.set_ylim(0, 1); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, fontsize=8)
    axes[0].set_ylabel("concorda com a resposta final")
    fig.tight_layout(); a = pasta / "fig_trajetorias.png"; fig.savefig(a, dpi=150); plt.close(fig); arqs.append(a)
    # 2) a faixa do tipo 2 de cada feature (primeiro alvo, features do fuzzy aditivo)
    chave = [k for k in fous if k[1] == "Fuzzy aditivo"][0]; fo = fous[chave]
    feats = list(dict.fromkeys(fo.feature)); fontes = [c.replace("pico em ", "") for c in fo.columns if c.startswith("pico em ")]
    fig, axes = plt.subplots(1, len(feats), figsize=(3.6 * len(feats), 3.2))
    axes = np.atleast_1d(axes)
    for ax, f in zip(axes, feats):
        sub = fo[fo.feature == f]
        Cs = [sub[f"pico em {s}"].values for s in fontes]
        lo_x, hi_x = min(c.min() for c in Cs), max(c.max() for c in Cs); pad = 0.15 * (hi_x - lo_x + 1e-9)
        xs = np.linspace(lo_x - pad, hi_x + pad, 400)
        Ms = np.stack([ft.pert_t1(xs[:, None], c[None, :])[:, 0, :] for c in Cs])   # (S, n, 3)
        for k, cor in zip(range(3), ["#2a78d6", "#7f8c8d", "#c0392b"]):
            ax.fill_between(xs, Ms[:, :, k].min(0), Ms[:, :, k].max(0), color=cor, alpha=.25, lw=0)
            for s in range(len(Cs)):
                ax.plot(xs, Ms[s, :, k], color=cor, lw=1, ls=["-", "--", ":", "-."][s % 4])
            ax.text(Cs[0][k], 1.04, ft.TERMOS[k], ha="center", fontsize=8, color=cor)
        ax.set_title(f, fontsize=10); ax.set_ylim(0, 1.12); ax.spines[["top", "right"]].set_visible(False)
    axes[0].set_ylabel("pertinência")
    fig.suptitle(f"Rede de fora: {chave[0]} · faixa do tipo 2 = discordância entre {' e '.join(fontes)} (linhas)", fontsize=10)
    fig.tight_layout(); a = pasta / "fig_faixas_tipo2.png"; fig.savefig(a, dpi=150); plt.close(fig); arqs.append(a)
    # 3) largura do intervalo × erro
    fig, axes = plt.subplots(1, len(preds), figsize=(4.2 * len(preds), 3.2), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (nome, p) in zip(axes, preds.items()):
        ax.scatter(p["Fuzzy aditivo: tipo 2 transferido"], p["Fuzzy aditivo t2 largura"], s=6, alpha=.35,
                   c=np.where(p.erro == 1, "#c0392b", "#2a78d6"))
        ax.set_title(f"rede de fora: {nome} (vermelho = erro)", fontsize=10); ax.set_xlabel("nota de erro (fuzzy aditivo tipo 2)")
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].set_ylabel("largura do intervalo"); fig.tight_layout()
    a = pasta / "fig_intervalos.png"; fig.savefig(a, dpi=150); plt.close(fig); arqs.append(a)
    log("figuras: " + ", ".join(x.name for x in arqs))
    return arqs


def salvar(cfg, infos, hists, tabs, res, zonas, regras, fous, preds):
    """Abas principais = só P, referências não fuzzy e fuzzy aditivo (o que vai para o artigo).
    Wang-Mendel fica em abas 'interno_*', só para consulta."""
    arq = Path(cfg.saida) / "resultados_tipo2.xlsx"
    princ, intern = res[res.método.isin(ft.PRINCIPAIS)], res[res.método.isin(ft.INTERNOS)]
    resumo = princ.pivot_table(index="método", columns="rede alvo", values="AUROC").reindex(ft.PRINCIPAIS)
    with pd.ExcelWriter(arq) as xw:
        pd.Series({k: str(v) for k, v in asdict(cfg).items()}).to_frame("valor").to_excel(xw, sheet_name="config")
        pd.DataFrame(infos).to_excel(xw, sheet_name="redes", index=False)
        resumo.round(4).to_excel(xw, sheet_name="resumo_AUROC")
        princ.round(4).to_excel(xw, sheet_name="deixa_uma_fora", index=False)
        zonas[zonas.sistema.str.startswith("Fuzzy aditivo")].round(4).to_excel(xw, sheet_name="zonas", index=False)
        for (alvo, fam), r in regras.items():
            if fam == "Fuzzy aditivo":
                r.to_excel(xw, sheet_name=f"regras_t2_{alvo}"[:31], index=False)
                fous[(alvo, fam)].round(4).to_excel(xw, sheet_name=f"faixas_{alvo}"[:31], index=False)
        for n, h in hists.items():
            if len(h):
                h.to_excel(xw, sheet_name=f"treino_{n}"[:31], index=False)
        for n, p in preds.items():
            p.drop(columns=[c for c in p.columns if c.startswith("Wang-Mendel")]).to_excel(xw, sheet_name=f"notas_{n}"[:31], index=False)
        # só para consulta
        intern.round(4).to_excel(xw, sheet_name="interno_wm_metricas", index=False)
        zonas[zonas.sistema.str.startswith("Wang-Mendel")].round(4).to_excel(xw, sheet_name="interno_wm_zonas", index=False)
        for (alvo, fam), r in regras.items():
            if fam == "Wang-Mendel":
                r.to_excel(xw, sheet_name=f"interno_wm_regras_{alvo}"[:31], index=False)
    return arq


def rodar_tudo(cfg, log=print):
    t0 = time.time()
    meta, cache = preparar_dados(cfg, log)
    infos, hists, tabs = [], {}, {}
    for nome in cfg.arquiteturas:
        rede, info, hist = treinar_ou_carregar(cfg, nome, meta, cache, log)
        tabs[nome] = ler_ou_carregar(cfg, nome, rede, info, meta, cache, log)
        infos.append(info); hists[nome] = hist
        del rede
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    res, zonas, regras, fous, preds = ft.deixa_uma_fora(tabs, lg.FEATURES, cfg.k_features, cfg.k_aditivo, cfg.forcar,
                                                        cfg.q_alarme, cfg.n_boot, log)
    figs = figuras(tabs, fous, preds, cfg.saida, log)
    arq = salvar(cfg, infos, hists, tabs, res, zonas, regras, fous, preds)
    log(f"pronto em {(time.time() - t0) / 60:.1f} min → {arq}")
    return {"meta": meta, "infos": infos, "hists": hists, "tabs": tabs, "res": res, "zonas": zonas,
            "regras": regras, "fous": fous, "preds": preds, "figuras": figs, "planilha": arq}

## Configuração

In [ ]:
import experimento as ex
cfg = ex.Config(
    metadata_csv='/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv',
    img_dirs=['/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1',
              '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'],
    saida='/kaggle/working/saida_tipo2',
    arquiteturas=['efficientnet_b0', 'resnet50', 'vit_b_16'],   # opcional: 'convnext_tiny'
    epocas=15, bs=32,
    pesos_locais={},   # sem internet: {'resnet50': '/kaggle/input/.../resnet50-11ad3fa6.pth', ...}
)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA — ligue a GPU em Settings → Accelerator')

## 1. Dados: split por lesão (a mesma lesão nunca aparece em dois conjuntos)

In [ ]:
import numpy as np, pandas as pd, warnings
from IPython.display import display, Image
warnings.filterwarnings('ignore'); pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
meta, cache = ex.preparar_dados(cfg)

## 2. Treinar cada rede e ler por dentro

Para cada rede: treino (ou checkpoint), limiar escolhido na validação, e leitura do teste com probes em todos os blocos.

In [ ]:
import fuzzy_t2 as ft, leitura_geral as lg
infos, hists, tabs = [], {}, {}
for nome in cfg.arquiteturas:
    rede, info, hist = ex.treinar_ou_carregar(cfg, nome, meta, cache)
    tabs[nome] = ex.ler_ou_carregar(cfg, nome, rede, info, meta, cache)
    infos.append(info); hists[nome] = hist
    del rede; torch.cuda.empty_cache() if torch.cuda.is_available() else None
display(pd.DataFrame(infos).round(3))

## 3. Deixa uma rede de fora: tipo 1 × tipo 2

Cada coluna é a rede que ficou de fora (o sistema nunca viu os erros dela).

| Método | O que é |
|---|---|
| Só P | a referência a superar |
| Regressão logística dentro | aprendida na própria rede (usa rótulos que, numa rede nova, não teríamos) |
| Regressão logística / boosting transferidos | aprendidos nas outras redes; referências não fuzzy |
| Fuzzy aditivo: tipo 1 dentro | o sistema fuzzy aprendido na própria rede (o melhor caso) |
| Fuzzy aditivo: tipo 1 transferido | regras e termos das outras redes |
| Fuzzy aditivo: tipo 1 + termos no alvo | regras das outras redes, termos ajustados com as features da rede nova (sem rótulos) |
| **Fuzzy aditivo: tipo 2 transferido** | termos = faixa entre as redes de origem |
| Fuzzy aditivo: tipo 2 + alvo no envelope | a faixa inclui também os termos da rede nova (sem rótulos) |

**O que conta como sucesso:** o tipo 2 transferido chega mais perto do "tipo 1 dentro" do que o tipo 1 transferido,
com o IC do ganho acima de zero; e a zona *incerto* concentra mais erros do que a zona *seguro*.

In [ ]:
res, zonas, regras, fous, preds = ft.deixa_uma_fora(tabs, lg.FEATURES, cfg.k_features, cfg.k_aditivo, cfg.forcar, cfg.q_alarme, cfg.n_boot)
princ = res[res.método.isin(ft.PRINCIPAIS)]
for met in ['AUROC', 'AUROC além de P', 'AURC (menor=melhor)']:
    print(met); display(princ.pivot_table(index='método', columns='rede alvo', values=met).reindex(ft.PRINCIPAIS).round(3))
display(princ[princ.método.str.contains(': tipo 2')][['rede alvo', 'método', 'ganho de AUROC sobre o tipo 1 transferido', 'IC 95% do ganho']])

### As três zonas do tipo 2 (limiar escolhido nas redes de origem)

In [ ]:
display(zonas[zonas.sistema.str.startswith('Fuzzy aditivo')].round(3))

## 4. As regras e as faixas

A tabela de faixas mostra, para cada termo, onde fica o pico em cada rede de origem: a distância entre eles é a incerteza do tipo 2.

In [ ]:
for (alvo, fam) in regras:
    if fam == 'Fuzzy aditivo':
        print(f'=== rede de fora: {alvo} ==='); display(fous[(alvo, fam)].round(3)); display(regras[(alvo, fam)])

## 5. Figuras

In [ ]:
figs = ex.figuras(tabs, fous, preds, cfg.saida)
for f in figs: display(Image(filename=str(f)))

## 6. Só para consulta: Wang-Mendel (fora do artigo)

In [ ]:
display(res[res.método.isin(ft.INTERNOS)].pivot_table(index='método', columns='rede alvo', values='AUROC').reindex(ft.INTERNOS).round(3))
display(zonas[zonas.sistema.str.startswith('Wang-Mendel')].round(3))

## 7. Salvar (mande este arquivo)

In [ ]:
arq = ex.salvar(cfg, infos, hists, tabs, res, zonas, regras, fous, preds)
print('Salvo:', arq)